## Emotion Recognition From Text

Scratch neural network (PyTorch) baseline using the Kaggle "Emotions dataset for NLP".

Goals
- Load train/val/test splits
- Build vocabulary and tokenize
- Train a BiLSTM classifier
- Evaluate with accuracy and macro-F1
- Prepare for transfer learning later

In [22]:
# Setup
import os
import random
import re
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_DIR = "archive"
TRAIN_PATH = os.path.join(DATA_DIR, "train.txt")
VAL_PATH = os.path.join(DATA_DIR, "val.txt")
TEST_PATH = os.path.join(DATA_DIR, "test.txt")

# Hyperparameters
max_vocab_size = 20000
max_len = 60
batch_size = 64
embedding_dim = 128
hidden_dim = 256
dropout = 0.5
lr = 1e-3
weight_decay = 1e-4
epochs = 15

Device: cpu


In [23]:
# Data loading

def load_split(path):
    texts, labels = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # split on last semicolon to be safe
            if ";" not in line:
                continue
            text, label = line.rsplit(";", 1)
            texts.append(text)
            labels.append(label)
    return texts, labels

train_texts, train_labels = load_split(TRAIN_PATH)
val_texts, val_labels = load_split(VAL_PATH)
test_texts, test_labels = load_split(TEST_PATH)

print("Train:", len(train_texts), "Val:", len(val_texts), "Test:", len(test_texts))

Train: 16000 Val: 2000 Test: 2000


In [24]:
# Label mapping
all_labels = sorted(set(train_labels))
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}

print("Labels:", all_labels)
print("Label count:", Counter(train_labels))

Labels: ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']
Label count: Counter({'joy': 5362, 'sadness': 4666, 'anger': 2159, 'fear': 1937, 'love': 1304, 'surprise': 572})


In [25]:
# Tokenization and vocabulary

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z']+|[0-9]+|[^\sA-Za-z0-9]", text)

special_tokens = ["<pad>", "<unk>"]

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

most_common = counter.most_common(max_vocab_size - len(special_tokens))
itos = special_tokens + [w for w, _ in most_common]
stoi = {w: i for i, w in enumerate(itos)}

pad_idx = stoi["<pad>"]
unk_idx = stoi["<unk>"]

print("Vocab size:", len(itos))
print("Sample tokens:", itos[:20])


def encode(text):
    tokens = tokenize(text)
    ids = [stoi.get(tok, unk_idx) for tok in tokens]
    if len(ids) < max_len:
        ids = ids + [pad_idx] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

Vocab size: 15214
Sample tokens: ['<pad>', '<unk>', 'i', 'feel', 'and', 'to', 'the', 'a', 'feeling', 'that', 'of', 'my', 'in', 'it', 'like', 'so', 'for', 'im', 'me', 'but']


In [26]:
# Dataset and DataLoaders

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(label2id[self.labels[idx]], dtype=torch.long)
        return x, y


train_ds = EmotionDataset(train_texts, train_labels)
val_ds = EmotionDataset(val_texts, val_labels)
test_ds = EmotionDataset(test_texts, test_labels)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

print("Batches (train/val/test):", len(train_loader), len(val_loader), len(test_loader))

Batches (train/val/test): 250 32 32


In [27]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
# Model: BiLSTM classifier with improvements

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx, dropout=0.5, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)  # Dropout after embedding
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0  # Dropout between LSTM layers
        )
        self.dropout = nn.Dropout(dropout)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths=None):
        emb = self.emb_dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        
        # Attention pooling (masked)
        mask = (x != pad_idx).unsqueeze(-1)  # [B, T, 1]
        attn_scores = self.attn(out)  # [B, T, 1]
        attn_scores = attn_scores.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)
        pooled = (out * attn_weights).sum(dim=1)
        
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        return logits

num_classes = len(all_labels)
model = BiLSTMClassifier(len(itos), embedding_dim, hidden_dim, num_classes, pad_idx, dropout, num_layers=2).to(device)

# Class weights to reduce class imbalance bias
label_counts = Counter(train_labels)
class_weights = torch.tensor(
    [1.0 / label_counts[id2label[i]] for i in range(num_classes)],
    dtype=torch.float
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [28]:
# Training and evaluation utilities

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        if is_train:
            optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(y.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

In [29]:
# Train loop with early stopping and scheduler
best_val_f1 = 0.0
best_state = None
patience = 4
patience_counter = 0

for epoch in range(1, epochs + 1):
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader)
    
    scheduler.step(val_f1)  # Adjust LR based on validation F1

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc * 100:.2f}% f1 {train_f1:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc * 100:.2f}% f1 {val_f1:.4f}"
    )
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print("Loaded best model by val macro-F1:", best_val_f1)

    # Save the best model
    torch.save(best_state, "best_model.pth")

Epoch 01 | train loss 1.7180 acc 22.66% f1 0.2079 | val loss 1.3540 acc 39.90% f1 0.4240
Epoch 02 | train loss 1.2408 acc 43.73% f1 0.4449 | val loss 0.8005 acc 68.30% f1 0.6881
Epoch 03 | train loss 0.8333 acc 63.21% f1 0.6363 | val loss 0.4603 acc 81.80% f1 0.8200
Epoch 04 | train loss 0.5860 acc 75.36% f1 0.7509 | val loss 0.3154 acc 86.80% f1 0.8575
Epoch 05 | train loss 0.4475 acc 82.03% f1 0.8097 | val loss 0.2542 acc 89.45% f1 0.8825
Epoch 06 | train loss 0.3642 acc 85.47% f1 0.8368 | val loss 0.2254 acc 90.50% f1 0.8874
Epoch 07 | train loss 0.3053 acc 88.31% f1 0.8632 | val loss 0.1876 acc 91.50% f1 0.8981
Epoch 08 | train loss 0.2583 acc 89.88% f1 0.8756 | val loss 0.1571 acc 92.50% f1 0.9044
Epoch 09 | train loss 0.2336 acc 90.61% f1 0.8828 | val loss 0.1569 acc 93.05% f1 0.9102
Epoch 10 | train loss 0.2065 acc 91.59% f1 0.8894 | val loss 0.1440 acc 93.20% f1 0.9108
Epoch 11 | train loss 0.1849 acc 92.46% f1 0.9003 | val loss 0.1500 acc 93.05% f1 0.9075
Epoch 12 | train loss

In [30]:
# Load the best model (can't train every time)
best_state = torch.load("best_model.pth")
model.load_state_dict(best_state)

<All keys matched successfully>

In [31]:
# Final evaluation on test
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(y.detach().cpu().tolist())

print("Test accuracy:", accuracy_score(all_labels, all_preds))
print("Test macro-F1:", f1_score(all_labels, all_preds, average="macro"))
print(classification_report(all_labels, all_preds, target_names=[id2label[i] for i in range(num_classes)]))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix:\n", cm)

Test accuracy: 0.9305
Test macro-F1: 0.9003498269525583
              precision    recall  f1-score   support

       anger       0.92      0.96      0.94       275
        fear       0.91      0.88      0.90       224
         joy       0.99      0.90      0.94       695
        love       0.74      0.99      0.84       159
     sadness       0.99      0.95      0.97       581
    surprise       0.68      1.00      0.81        66

    accuracy                           0.93      2000
   macro avg       0.87      0.95      0.90      2000
weighted avg       0.94      0.93      0.93      2000

Confusion matrix:
 [[263  11   1   0   0   0]
 [  1 198   0   0   2  23]
 [  5   0 627  54   1   8]
 [  0   0   2 157   0   0]
 [ 18   8   3   2 550   0]
 [  0   0   0   0   0  66]]


In [ ]:
def predict_emotion(text):
    model.eval()
    x = torch.tensor([encode(text)], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()
    return id2label[pred]

print("Example:", predict_emotion("I am afraid of the dark."))
# print("Example:", predict_emotion(""))

Example: fear
